# Phase 2 — Feature Engineering

This notebook covers the experimentation and development of the feature engineering pipeline.

## 2.1 Time Features

In [12]:
import pandas as pd
import numpy as np
import os
import sys
import importlib

# Add project root to path
if os.getcwd().endswith('notebooks'):
    project_root = os.path.dirname(os.getcwd())
    if project_root not in sys.path:
        sys.path.append(project_root)

import src.utils.data_loader
import src.utils.features
importlib.reload(src.utils.data_loader)
importlib.reload(src.utils.features)

from src.utils.data_loader import load_raw
from src.utils.features import create_time_features

print("--- Prototyping Time Features ---")
df = load_raw()
df = create_time_features(df)

print("Time features extracted successfully in the notebook.")
df[['Time_of_Booking', 'hour', 'hour_sin', 'hour_cos', 'is_night', 'is_rush_hour']].head()

INFO:src.utils.logger:Loading raw data from c:\Users\BRAIN\Desktop\ML\dynamic_pricing\data/raw/dynamic_pricing.csv


--- Prototyping Time Features ---
Time features extracted successfully in the notebook.


,Time_of_Booking,hour,hour_sin,hour_cos,is_night,is_rush_hour
0,Night,2,0.500000,0.866025,1,0
1,Evening,19,-0.965926,0.258819,0,1
2,Afternoon,14,-0.500000,-0.866025,0,0
3,Afternoon,14,-0.500000,-0.866025,0,0
4,Afternoon,14,-0.500000,-0.866025,0,0


## 2.2 Demand–Supply Features

In [13]:
import importlib
import src.utils.features
importlib.reload(src.utils.features)

from src.utils.features import create_demand_supply_features

print("--- Prototyping Demand-Supply Features ---")
df = create_demand_supply_features(df, is_training=True)

print("Demand-supply features extracted successfully.")
df[['Number_of_Riders', 'Number_of_Drivers', 'Location_Category', 'hour', 'demand_supply_ratio', 'hist_demand_supply_ratio_loc_hour']].head()

--- Prototyping Demand-Supply Features ---
Demand-supply features extracted successfully.


,Number_of_Riders,Number_of_Drivers,Location_Category,hour,demand_supply_ratio,hist_demand_supply_ratio_loc_hour
0,90,45,Urban,2,1.956522,2.921795
1,58,39,Suburban,19,1.450000,2.688637
2,42,31,Rural,14,1.312500,2.904643
3,89,28,Rural,14,3.068966,2.904643
4,78,22,Rural,14,3.391304,2.904643


## 2.3 Customer Features

In [14]:
from src.utils.features import create_customer_features
importlib.reload(src.utils.features)

print("--- Prototyping Customer Features ---")
df = create_customer_features(df)

print("Customer features extracted successfully.")
df[['Customer_Loyalty_Status', 'loyalty_numeric', 'Number_of_Past_Rides', 'ride_tenure_bucket', 'Average_Ratings', 'rating_bucket', 'is_new_user', 'is_high_value']].head()

--- Prototyping Customer Features ---
Customer features extracted successfully.


,Customer_Loyalty_Status,loyalty_numeric,Number_of_Past_Rides,ride_tenure_bucket,Average_Ratings,rating_bucket,is_new_user,is_high_value
0,Silver,1,13,6-20,4.47,4-5,0,0
1,Silver,1,72,51-100,4.06,4-5,0,0
2,Silver,1,0,0-5,3.99,3-4,1,0
3,Regular,0,67,51-100,4.31,4-5,0,0
4,Regular,0,74,51-100,3.77,3-4,0,0


## 2.4 Categorical Encoding

In [15]:
from src.utils.features import create_categorical_features
importlib.reload(src.utils.features)

print("--- Prototyping Categorical Encoding ---")
df = create_categorical_features(df, is_training=True)

print("Categorical features encoded successfully.")
df[['Vehicle_Type', 'vehicle_Economy', 'vehicle_Premium', 'Location_Category', 'location_score']].head()

--- Prototyping Categorical Encoding ---
Categorical features encoded successfully.


,Vehicle_Type,vehicle_Economy,vehicle_Premium,Location_Category,location_score
0,Premium,0.0,1.0,Urban,363.698642
1,Economy,1.0,0.0,Suburban,374.313861
2,Premium,0.0,1.0,Rural,379.919861
3,Premium,0.0,1.0,Rural,379.919861
4,Economy,1.0,0.0,Rural,379.919861


## 2.5 Interaction Features

In [16]:
from src.utils.features import create_interaction_features
importlib.reload(src.utils.features)

print("--- Prototyping Interaction Features ---")
df = create_interaction_features(df)

print("Interaction features extracted successfully.")
df[['duration_x_vehicle_premium', 'demand_x_loyalty', 'rating_x_tenure', 'location_x_rush_hour']].head()

--- Prototyping Interaction Features ---
Interaction features extracted successfully.


,duration_x_vehicle_premium,demand_x_loyalty,rating_x_tenure,location_x_rush_hour
0,90.0,1.956522,58.11,0.000000
1,0.0,1.450000,292.32,374.313861
2,76.0,1.312500,0.00,0.000000
3,134.0,0.000000,288.77,0.000000
4,0.0,0.000000,278.98,0.000000


## 2.6 Scaling & Normalization

In [17]:
from src.utils.features import scale_features
importlib.reload(src.utils.features)

print("--- Prototyping Scaling & Normalization ---")
df = scale_features(df, is_training=True)

print("Scaling completed successfully.")
df[['Number_of_Riders', 'Number_of_Drivers', 'Average_Ratings', 'Expected_Ride_Duration', 'Historical_Cost_of_Ride']].head()

--- Prototyping Scaling & Normalization ---
Scaling completed successfully.


,Number_of_Riders,Number_of_Drivers,Average_Ratings,Expected_Ride_Duration,Historical_Cost_of_Ride
0,0.731707,0.851852,0.262295,-0.144144,-0.268950
1,-0.048780,0.629630,-0.275410,-0.708709,-0.650722
2,-0.439024,0.333333,-0.367213,-0.312312,-0.111451
3,0.707317,0.222222,0.052459,0.384384,0.374160
4,0.439024,0.000000,-0.655738,0.564565,0.752811


## 2.7 End-to-End Pipeline

In [18]:
from src.utils.features import build_feature_pipeline
from src.utils.data_loader import load_raw
importlib.reload(src.utils.features)

print("--- Verifying End-to-End Pipeline ---")
df_raw = load_raw()

# Simulate split
train_size = int(0.7 * len(df_raw))
val_size = int(0.15 * len(df_raw))

train_df = df_raw.iloc[:train_size]
val_df = df_raw.iloc[train_size:train_size + val_size]
test_df = df_raw.iloc[train_size + val_size:]

X_train, X_val, X_test, y_train, y_val, y_test = build_feature_pipeline(
    train_df, val_df, test_df
)

print(f"Pipeline complete.")
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

# Save processed data
processed_path = os.path.join(project_root, 'data', 'processed', 'dynamic_pricing_processed.csv')
X_train.join(y_train).to_csv(processed_path, index=False)
print(f"Final feature matrix saved to {processed_path}")

INFO:src.utils.logger:Loading raw data from c:\Users\BRAIN\Desktop\ML\dynamic_pricing\data/raw/dynamic_pricing.csv


--- Verifying End-to-End Pipeline ---
Pipeline complete.
X_train shape: (700, 32), y_train shape: (700,)
X_val shape: (150, 32), y_val shape: (150,)
X_test shape: (150, 32), y_test shape: (150,)
Final feature matrix saved to c:\Users\BRAIN\Desktop\ML\dynamic_pricing\data\processed\dynamic_pricing_processed.csv
